In [4]:
torch.cuda.is_available()

True

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [2]:
#下载函数
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertModel, BertTokenizer,BertConfig
from sklearn.metrics import confusion_matrix, classification_report
from transformers import BertTokenizer, BertConfig, Trainer, TrainingArguments
from transformers import BertForSequenceClassification
import numpy as np
import evaluate
import torch
import torch.nn as nn
import math
#本地函数
from MyBertSelfAttention_alltoken import MyBertSelfAttention, MyBertEncoder, MyBertLayer, MyBertModel, MyBertForSequenceClassification

from ear import compute_negative_entropy
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
import torch.nn.functional as F

### WS dataset

In [4]:
import os, glob
import pandas as pd
from datasets import Dataset, Value

ROOT = os.path.expanduser("~/test/hate-speech-dataset-master")
ALL_DIR  = os.path.join(ROOT, "all_files")
TEST_DIR = os.path.join(ROOT, "sampled_test")
CSV_PATH = os.path.join(ROOT, "annotations_metadata.csv")

# 1) 读标注表
df = pd.read_csv(CSV_PATH)

# 自动找到 label 列（优先叫 label；否则用最后一列）
label_col = "label" if "label" in df.columns else df.columns[-1]

# 统一类型
df["file_id"] = df["file_id"].astype(str)
df[label_col] = df[label_col].astype(str)

# 2) label -> int（0/1）
label_map = {"noHate": 0, "hate": 1}
df["label"] = df[label_col].map(label_map)

# 丢掉没映射成功的行（比如有空值/拼写不同）
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype("int64")

# 3) 读文本：每个 file_id 对应 all_files/file_id.txt
def read_txt(file_id: str) -> str:
    path = os.path.join(ALL_DIR, f"{file_id}.txt")
    if not os.path.exists(path):
        return ""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

df["tweet"] = df["file_id"].apply(read_txt)
df = df[df["tweet"].str.len() > 0].reset_index(drop=True)

# 4) 用 sampled_test 里的文件名做 test split
test_stems = set(os.path.splitext(os.path.basename(p))[0]
                 for p in glob.glob(os.path.join(TEST_DIR, "*.txt")))

is_test = df["file_id"].isin(test_stems)
train_df = df[~is_test][["tweet", "label"]].reset_index(drop=True)
test_df  = df[ is_test][["tweet", "label"]].reset_index(drop=True)

# 5) 转 HF Dataset，并强制 label 为 int64（避免 CE loss 报错）
train_dataset = Dataset.from_pandas(train_df, preserve_index=False).cast_column("label", Value("int64"))
test_dataset  = Dataset.from_pandas(test_df,  preserve_index=False).cast_column("label", Value("int64"))

print("train label counts:", train_df["label"].value_counts().to_dict())
print("test  label counts:", test_df["label"].value_counts().to_dict())
print(train_dataset, test_dataset)


Casting the dataset: 100%|██████████| 478/478 [00:00<00:00, 251552.99 examples/s]

train label counts: {0: 9268, 1: 957}
test  label counts: {1: 239, 0: 239}
Dataset({
    features: ['tweet', 'label'],
    num_rows: 10225
}) Dataset({
    features: ['tweet', 'label'],
    num_rows: 478
})


### 在训练集上跑

In [8]:
import re
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# 0) 准备 docs
train_docs = train_df["tweet"].astype(str).tolist()
test_docs  = test_df["tweet"].astype(str).tolist()

# 1) （可选）轻量清洗：去URL/@user，多余空格
def clean_tweet(t: str) -> str:
    t = re.sub(r"http\S+|www\.\S+", " ", t)   # urls
    t = re.sub(r"@\w+", " ", t)              # mentions
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_docs = [clean_tweet(t) for t in train_docs]
test_docs  = [clean_tweet(t) for t in test_docs]

# 2) 自定义向量器：控制 ngram / stopwords / 词表门槛
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=4
)

# 3) 建模
# language="english" 时默认 embedding 模型一般是 all-MiniLM-L6-v2（短文本聚类很常用）:contentReference[oaicite:2]{index=2}
topic_model = BERTopic(
    language="english",
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(train_docs)

# 4) 查看总览
topic_info = topic_model.get_topic_info()   # 含 Topic、Count、Name 等 :contentReference[oaicite:3]{index=3}
print(topic_info.head(10))

# 5) 导出“每个 topic 的 top 词”
top_n = 15
rows = []
for topic_id in topic_info["Topic"].tolist():
    if topic_id == -1:   # -1 通常是离群点/未分配文档
        continue
    words_weights = topic_model.get_topic(topic_id)  # [(word, weight), ...] :contentReference[oaicite:4]{index=4}
    for rank, (w, score) in enumerate(words_weights[:top_n], start=1):
        rows.append({"topic": topic_id, "rank": rank, "word": w, "weight": score})

topic_words_df = pd.DataFrame(rows).sort_values(["topic", "rank"])
topic_words_df.to_csv("bertopic_topic_words.csv", index=False, encoding="utf-8-sig")
print("Saved:", "bertopic_topic_words.csv")


2026-01-31 16:30:07,395 - BERTopic - Embedding - Transforming documents to embeddings.


KeyboardInterrupt: 

In [15]:
test_topics, test_probs = topic_model.transform(test_docs)  # :contentReference[oaicite:5]{index=5}

test_out = test_df.copy()
test_out["pred_topic"] = test_topics
test_out.to_csv("test_with_topics.csv", index=False, encoding="utf-8-sig")
print("Saved:", "test_with_topics.csv")


Batches: 100%|██████████| 15/15 [00:19<00:00,  1.29s/it]
2025-12-27 21:26:44,426 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-12-27 21:26:46,493 - BERTopic - Dimensionality - Completed ✓
2025-12-27 21:26:46,494 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-12-27 21:26:46,532 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2025-12-27 21:26:47,085 - BERTopic - Probabilities - Completed ✓
2025-12-27 21:26:47,086 - BERTopic - Cluster - Completed ✓


Saved: test_with_topics.csv


In [1]:
import sys
!{sys.executable} -m pip install -U nbformat ipython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 11.0 MB/s  0:00:00
  Attempting uninstall: ipython
    Found existing installation: ipython 9.2.0
    Uninstalling ipython-9.2.0:
      Successfully uninstalled ipython-9.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [nbformat]6/7 [nbformat]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [9]:
fig = topic_model.visualize_documents(train_docs)
fig.show()


In [7]:
print(topic_model.embedding_model)


In [11]:
# topics 是 fit_transform(train_docs) 的输出
train_df_with_topic = train_df.copy()
train_df_with_topic["topic"] = topics

# topic 0 的随机 10 条(white)
train_df_with_topic[train_df_with_topic["topic"] == 9].sample(10, random_state=0)["tweet"].tolist()



['If I have to start compartmentalizing I use Nordic and Heathen',
 "I'm curious ( but doubtful ) ; is there anyone here who 's expressed SUPPORT for Nordic superiority that is not Nordic himself ?",
 "It makes me so damn angry that the swedes ca n't see it !!!",
 'Similar with Norway I think .',
 "Don't start me raging on all the German and Swedish students we have here taking places from the country 's native people I can imagine there is quite alot Finns in Swedens universities though .",
 'I think anyone who wants to give up the Swedish welfare state is an enemy of Sweden and certainly no nationalist .',
 'Sørensen , 19 years old , was born April 8 , 1982 in Kolding .',
 'hope one day i will be able to see finland withouth those * * * * ing monkey ass soumalian ... im glad .. because finland has lowest immigrate in europe ..... kick those * * * * ing soumalians ass .....',
 "Nordic is not someone 's heritage it 's a sub race category than one person 's heritage will be put into .",

In [13]:
# 看离群点 -1 的随机 10 条
train_df_with_topic[train_df_with_topic["topic"] == -1].sample(10, random_state=1)["tweet"].tolist()

['Many of them are doing just as their masters desire .',
 'I run three miles a day during the week and day hike with my rifle and ALICE gear on the weekends .',
 'Perhaps they realize people are waking up to the story and figure they should take their gains and quit while they are ahead .',
 'Ouuuuuuu , do I hate !',
 'The only thing holding us back are our traitors and the 40billion anti-european people of the earth that flourished on our technology .',
 'Reproducing does ....',
 'Millions upon millions of them , until there is no British culture left .',
 "OMG unfortunately they are very good at hot air. did n't see any massive demos to prosecute the kiddie fiddlers amongst the Christian Brothers .",
 'To love ones people in these dark ages is considered a weakness to be exploited .',
 'White Power Brother !']

### 在全体数据上进行Topic提取

In [9]:
import re
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from umap import UMAP

# === 0) 合并“全体数据”（train + test）===
all_df = pd.concat([train_df, test_df], ignore_index=True)

# all_df = all_df.drop_duplicates(subset=["tweet"]).reset_index(drop=True)

# === 1) 轻量清洗：去 URL / @mention / 多余空格 ===
def clean_tweet(t: str) -> str:
    t = re.sub(r"http\S+|www\.\S+", " ", str(t))  # urls
    t = re.sub(r"@\w+", " ", t)                  # mentions
    t = re.sub(r"\s+", " ", t).strip()
    return t

all_df["tweet_clean"] = all_df["tweet"].apply(clean_tweet)

# === 2) CountVectorizer：控制主题词候选（1-2gram，去停用词）===
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)


In [10]:
umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=2
)
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=2,
    cluster_selection_method="eom",
    prediction_data=True
)
# === 3) 在全体数据上训练 BERTopic ===
topic_model_all = BERTopic(
    language="english",
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    #nr_topics=50,
    calculate_probabilities=False,  # 只做主题分析一般不需要 probs，省内存
    verbose=True
)

docs_all = all_df["tweet_clean"].tolist()
topics_all, _ = topic_model_all.fit_transform(docs_all)

# 把 topic 写回每条文档
all_df["topic_all"] = topics_all

# === 4) 计算每个 topic 的 hate 比例并排序 ===
# hate_ratio = 该topic中 label==1 的比例（因为 label 是 0/1，所以 mean 就是比例）
topic_stats = (
    all_df.groupby("topic_all")["label"]
    .agg(count="size", hate_ratio="mean")
    .reset_index()
    .rename(columns={"topic_all": "Topic"})
)

topic_info_all = topic_model_all.get_topic_info()  # Topic / Count / Name 等
topic_ranked = (
    topic_info_all.merge(topic_stats, on="Topic", how="left")
    .sort_values(["hate_ratio", "count"], ascending=[False, False])
    .reset_index(drop=True)
)

print(topic_ranked.head(20))

# === 5) 导出：topic 排名表（含 hate_ratio）===
topic_ranked.to_csv("topics_ranked_by_hate_ratio_all.csv", index=False, encoding="utf-8-sig")
print("Saved: topics_ranked_by_hate_ratio_all.csv")

# === 6) 导出：每个 topic 的 top 词（全体模型）===
top_n = 15
rows = []
for topic_id in topic_ranked["Topic"].tolist():
    if topic_id == -1:  # -1 是离群点集合，通常不导出
        continue
    words_weights = topic_model_all.get_topic(topic_id) or []
    for rank, (w, score) in enumerate(words_weights[:top_n], start=1):
        rows.append({"topic": topic_id, "rank": rank, "word": w, "weight": score})

topic_words_all = pd.DataFrame(rows).sort_values(["topic", "rank"])
topic_words_all.to_csv("bertopic_topic_words_all.csv", index=False, encoding="utf-8-sig")
print("Saved: bertopic_topic_words_all.csv")


2026-01-28 19:05:04,999 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 335/335 [00:02<00:00, 123.53it/s]
2026-01-28 19:05:10,133 - BERTopic - Embedding - Completed ✓
2026-01-28 19:05:10,133 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-28 19:05:37,069 - BERTopic - Dimensionality - Completed ✓
2026-01-28 19:05:37,071 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-28 19:05:37,585 - BERTopic - Cluster - Completed ✓
2026-01-28 19:05:37,589 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-28 19:05:37,809 - BERTopic - Representation - Completed ✓


    Topic  Count                                   Name  \
0      41     58             41_apes_ape_monkeys_monkey   
1       7    183           7_black_negro_blacks_negroes   
2      29     76        29_africa_south_south africa_sa   
3      46     54           46_asians_asian_white_whites   
4       6    290               6_jews_jew_jewish_israel   
5      38     61          38_border_mexico_indians_drug   
6      28     76    28_ukraine_egyptians_muslim_muslims   
7      24     81  24_whites_white_white people_minority   
8       2    312    2_country_people_government_welfare   
9       8    179       8_ireland_irish_northern_country   
10     32     69    32_london_immigrants_canada_british   
11     47     51       47_canada_canadians_justin_prime   
12     40     58       40_census_whites_town_population   
13     33     69    33_holocaust_hitler_germans_youtube   
14     39     60       39_race_mixing_races_race mixing   
15     27     77      27_white_kids_white kids_children 

In [11]:
topic_info = topic_model_all.get_topic_info()   # 含 Topic/Count/Name 等

non_topics = (topic_info["Topic"] == -1).sum()
# 1) 最终 topic 数量（不含 -1）
n_topics = (topic_info["Topic"] != -1).sum()

# 2) 去掉样本数 < 10 的 topic（同时排除 -1）
topic_info_ge10 = topic_info[(topic_info["Topic"] != -1) & (topic_info["Count"] >= 10)]

# 3) 过滤后的 topic 数量
n_topics_ge10 = len(topic_info_ge10)
#print("Topics -1:", non_topics)
print("Total topics (exclude -1):", n_topics)
print("Topics with >=10 docs:", n_topics_ge10)

# 可选：看一下被过滤掉的小topic有哪些
small_topics = topic_info[(topic_info["Topic"] != -1) & (topic_info["Count"] < 30)]
print("Small topics (<10):", len(small_topics))


Total topics (exclude -1): 48
Topics with >=10 docs: 48
Small topics (<10): 0


In [12]:
# 统计离群点占比
import pandas as pd

def summarize_topics(model, topics, name=""):
    info = model.get_topic_info()
    n_topics = (info["Topic"] != -1).sum()
    n_outliers = (pd.Series(topics) == -1).sum()
    print(f"[{name}] topics(excl -1) = {n_topics}, outliers(-1) = {n_outliers} ({n_outliers/len(topics):.1%})")
    return info

info_before = summarize_topics(topic_model_all, topics_all, "before")


[before] topics(excl -1) = 48, outliers(-1) = 4410 (41.2%)


In [26]:
topic_model_all.reduce_topics(docs_all, nr_topics="auto")
topics_reduced = topic_model_all.topics_   # 融合后的新 topic 分配
info_reduced = summarize_topics(topic_model_all, topics_reduced, "after reduce_topics(auto)")


2026-01-28 15:36:16,077 - BERTopic - Topic reduction - Reducing number of topics
2026-01-28 15:36:16,139 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-28 15:36:16,359 - BERTopic - Representation - Completed ✓
2026-01-28 15:36:16,361 - BERTopic - Topic reduction - Reduced number of topics from 180 to 21


[after reduce_topics(auto)] topics(excl -1) = 20, outliers(-1) = 4875 (45.5%)


In [13]:
hier = topic_model_all.hierarchical_topics(docs_all)
fig = topic_model_all.visualize_hierarchy(hierarchical_topics=hier)
fig.show()

100%|██████████| 47/47 [00:00<00:00, 403.97it/s]

In [20]:
topic_model_all.visualize_documents(
    docs_all,
    topics=topics_all,      # 可选，不传也行
    embeddings=None     # 默认用模型内部 embedding
)

In [15]:
import sys
!{sys.executable} -m plotly_get_chrome


/home/fang/venv/bin/python: No module named plotly_get_chrome


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


### 仅对hate标签的数据进行topic提取

In [22]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=2
)
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=1,
    cluster_selection_method="eom",
    prediction_data=True
)

hate_df = all_df[all_df["label"] == 1].reset_index(drop=True)
docs_hate = hate_df["tweet_clean"].tolist()

topic_model_hate = BERTopic(
    language="english",
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    #nr_topics=10,
    calculate_probabilities=False,
    verbose=True
)

topics_hate, _ = topic_model_hate.fit_transform(docs_hate)
hate_df["topic_hate_only"] = topics_hate

topic_info_hate = topic_model_hate.get_topic_info()
print(topic_info_hate.head(20))

# 导出 hate-only 的 topic info
topic_info_hate.to_csv("topic_info_hate_only.csv", index=False, encoding="utf-8-sig")
print("Saved: topic_info_hate_only.csv")

# 导出 hate-only 的 topic 词
top_n = 10
rows = []
for topic_id in topic_info_hate["Topic"].tolist():
    if topic_id == -1:
        continue
    words_weights = topic_model_hate.get_topic(topic_id) or []
    for rank, (w, score) in enumerate(words_weights[:top_n], start=1):
        rows.append({"topic": topic_id, "rank": rank, "word": w, "weight": score})

topic_words_hate = pd.DataFrame(rows).sort_values(["topic", "rank"])
topic_words_hate.to_csv("bertopic_topic_words_hate_only.csv", index=False, encoding="utf-8-sig")
print("Saved: bertopic_topic_words_hate_only.csv")


2026-01-31 16:51:24,426 - BERTopic - Embedding - Transforming documents to embeddings.
Batches:   6%|▋         | 2/32 [00:00<00:00, 62.78it/s]


TypeError: 'float' object is not subscriptable

In [17]:
topic_info = topic_model_hate.get_topic_info()   # 含 Topic/Count/Name 等

# 1) 最终 topic 数量（不含 -1）
n_topics = (topic_info["Topic"] != -1).sum()

# 2) 去掉样本数 < 10 的 topic（同时排除 -1）
topic_info_ge10 = topic_info[(topic_info["Topic"] != -1) & (topic_info["Count"] >= 10)]

# 3) 过滤后的 topic 数量
n_topics_ge10 = len(topic_info_ge10)

print("Total topics (exclude -1):", n_topics)
print("Topics with >=10 docs:", n_topics_ge10)

# 可选：看一下被过滤掉的小topic有哪些
small_topics = topic_info[(topic_info["Topic"] != -1) & (topic_info["Count"] < 10)]
print("Small topics (<10):", len(small_topics))
# 统计离群点占比


info_before = summarize_topics(topic_model_hate, topics_hate, "before")


Total topics (exclude -1): 8
Topics with >=10 docs: 8
Small topics (<10): 0
[before] topics(excl -1) = 8, outliers(-1) = 371 (31.0%)


In [18]:
hier = topic_model_hate.hierarchical_topics(docs_hate)
fig = topic_model_hate.visualize_hierarchy(hierarchical_topics=hier)
fig.show()

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00, 428.33it/s]


In [19]:
topic_model_hate.visualize_documents(
    docs_hate,
    topics=topics_hate,      # 可选，不传也行
    embeddings=None     # 默认用模型内部 embedding
)

### 将topic信息concat在text后

In [6]:
import re
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# ========== A) 合并全体数据（train+test）==========
all_df = pd.concat([train_df, test_df], ignore_index=True)

# ========== B) 轻量清洗（和你之前一致即可）==========
def clean_tweet(t: str) -> str:
    t = re.sub(r"http\S+|www\.\S+", " ", str(t))
    t = re.sub(r"@\w+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

all_df["tweet_clean"] = all_df["tweet"].apply(clean_tweet)

docs_all = all_df["tweet_clean"].tolist()

# ========== C) BERTopic（可按你之前参数）==========
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

topic_model_all = BERTopic(
    language="english",
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)

topics_all, _ = topic_model_all.fit_transform(docs_all)
all_df["topic_id"] = topics_all

# ========== D) 构建 topic -> name/representation 映射 ==========
topic_info = topic_model_all.get_topic_info()  # Topic/Count/Name 等
topicid_to_name = dict(zip(topic_info["Topic"], topic_info["Name"]))

TOP_N = 15
topicid_to_repr = {}
for tid in topic_info["Topic"].tolist():
    if tid == -1:
        topicid_to_repr[tid] = ""  # 离群点不给关键词
        continue
    ww = topic_model_all.get_topic(tid) or []
    words = [w for (w, _) in ww[:TOP_N]]
    topicid_to_repr[tid] = ", ".join(words)

all_df["topic_name"] = all_df["topic_id"].map(topicid_to_name)
all_df["topic_repr"] = all_df["topic_id"].map(topicid_to_repr)

# ========== E) 生成给 BERT 用的输入文本（把topic关键词拼到原文后）==========
# 你也可以不拼，直接用 tweet_clean 做 baseline
all_df["text_for_bert"] = all_df.apply(
    lambda r: r["tweet_clean"] if (r["topic_id"] == -1 or r["topic_repr"] == "")
    else f'{r["tweet_clean"]} [TOPIC] {r["topic_repr"]}',
    axis=1
)

# ========== F) 导出新 CSV（含原文 + label + topic信息）==========
cols = ["tweet", "tweet_clean", "label", "topic_id", "topic_name", "topic_repr", "text_for_bert"]
all_df_out = all_df[cols].copy()
all_df_out.to_csv("WS_with_topics.csv", index=False, encoding="utf-8-sig")
print("Saved: WS_with_topics.csv")

# 如果你也想分别导出 train/test（保持你原始 split）
#train_with_topics = all_df_out.iloc[:len(train_df)].copy()
#test_with_topics  = all_df_out.iloc[len(train_df):].copy()
#train_with_topics.to_csv("train_with_topics.csv", index=False, encoding="utf-8-sig")
#test_with_topics.to_csv("test_with_topics.csv", index=False, encoding="utf-8-sig")
#print("Saved: train_with_topics.csv, test_with_topics.csv")


2026-01-09 14:52:36,514 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 335/335 [00:04<00:00, 83.24it/s] 
2026-01-09 14:52:44,459 - BERTopic - Embedding - Completed ✓
2026-01-09 14:52:44,460 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-09 14:53:01,854 - BERTopic - Dimensionality - Completed ✓
2026-01-09 14:53:01,856 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-09 14:53:02,353 - BERTopic - Cluster - Completed ✓
2026-01-09 14:53:02,358 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-09 14:53:02,651 - BERTopic - Representation - Completed ✓


Saved: WS_with_topics.csv


In [7]:
train_with_topics = all_df_out.iloc[:len(train_df)].copy()
test_with_topics  = all_df_out.iloc[len(train_df):].copy()
train_with_topics.to_csv("train_with_topics.csv", index=False, encoding="utf-8-sig")
test_with_topics.to_csv("test_with_topics.csv", index=False, encoding="utf-8-sig")
print("Saved: train_with_topics.csv, test_with_topics.csv")

Saved: train_with_topics.csv, test_with_topics.csv


使用

In [8]:
import numpy as np
import evaluate
from datasets import Dataset, Value
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

def train_bert_classifier(train_df_in, test_df_in, text_col="tweet_clean", out_dir="bert_out"):
    # 1) 转 HF Dataset
    tr = train_df_in[[text_col, "label"]].rename(columns={text_col: "text"}).copy()
    te = test_df_in[[text_col, "label"]].rename(columns={text_col: "text"}).copy()

    tr["label"] = tr["label"].astype("int64")
    te["label"] = te["label"].astype("int64")

    train_ds = Dataset.from_pandas(tr, preserve_index=False).cast_column("label", Value("int64"))
    test_ds  = Dataset.from_pandas(te, preserve_index=False).cast_column("label", Value("int64"))

    # 2) tokenizer
    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

    train_ds = train_ds.map(tokenize, batched=True)
    test_ds  = test_ds.map(tokenize, batched=True)

    train_ds = train_ds.remove_columns(["text"])
    test_ds  = test_ds.remove_columns(["text"])
    train_ds.set_format("torch")
    test_ds.set_format("torch")

    # 3) model
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    # 4) training args
    args = TrainingArguments(
        output_dir=out_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    print("device:", trainer.args.device, "n_gpu:", trainer.args.n_gpu)
    trainer.train()
    metrics = trainer.evaluate()
    return trainer, metrics

# ========== 运行：baseline（只用 tweet_clean）==========
trainer_base, metrics_base = train_bert_classifier(
    train_with_topics, test_with_topics,
    text_col="tweet_clean",
    out_dir="bert_baseline"
)
print("Baseline metrics:", metrics_base)

# ========== 运行：topic-augmented（用 text_for_bert）==========
trainer_topic, metrics_topic = train_bert_classifier(
    train_with_topics, test_with_topics,
    text_col="text_for_bert",
    out_dir="bert_with_topics"
)
print("With-topic metrics:", metrics_topic)


Map: 100%|██████████| 478/478 [00:00<00:00, 11938.91 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


device: cuda:0 n_gpu: 1


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.228500,0.751880,0.702929,0.679981
2,0.109500,1.006570,0.709205,0.689295
3,0.072400,1.273451,0.736402,0.724836


Baseline metrics: {'eval_loss': 1.2734507322311401, 'eval_accuracy': 0.7364016736401674, 'eval_f1_macro': 0.7248355263157895, 'eval_runtime': 0.8646, 'eval_samples_per_second': 552.867, 'eval_steps_per_second': 17.349, 'epoch': 3.0}


Map: 100%|██████████| 478/478 [00:00<00:00, 11443.76 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


device: cuda:0 n_gpu: 1


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.228300,0.671758,0.748954,0.736977
2,0.108500,0.988635,0.738494,0.724699
3,0.052100,1.162700,0.744770,0.734042


With-topic metrics: {'eval_loss': 0.6717581748962402, 'eval_accuracy': 0.7489539748953975, 'eval_f1_macro': 0.7369772560528247, 'eval_runtime': 0.8713, 'eval_samples_per_second': 548.609, 'eval_steps_per_second': 17.216, 'epoch': 3.0}


In [19]:
import numpy as np
import pandas as pd
from datasets import Dataset, Value

# ---------- 1) 指标对比表（metrics table） ----------
def pick(m: dict, key: str):
    return m.get(key, np.nan)

metrics_table = pd.DataFrame([
    {
        "run": "baseline(tweet_clean)",
        "eval_loss": pick(metrics_base, "eval_loss"),
        "accuracy": pick(metrics_base, "eval_accuracy"),
        "f1_macro": pick(metrics_base, "eval_f1_macro"),
        "runtime_s": pick(metrics_base, "eval_runtime"),
        "samples_per_second": pick(metrics_base, "eval_samples_per_second"),
    },
    {
        "run": "with_topic(text_for_bert)",
        "eval_loss": pick(metrics_topic, "eval_loss"),
        "accuracy": pick(metrics_topic, "eval_accuracy"),
        "f1_macro": pick(metrics_topic, "eval_f1_macro"),
        "runtime_s": pick(metrics_topic, "eval_runtime"),
        "samples_per_second": pick(metrics_topic, "eval_samples_per_second"),
    },
])

print(metrics_table)
metrics_table.to_csv("metrics_compare_baseline_vs_topic.csv", index=False, encoding="utf-8-sig")
print("Saved: metrics_compare_baseline_vs_topic.csv")


# ---------- 2) 预测写回 CSV（逐条对比） ----------
def build_pred_dataset(df_in: pd.DataFrame, text_col: str, tokenizer, max_length: int = 128) -> Dataset:
    te = df_in[[text_col, "label"]].rename(columns={text_col: "text"}).copy()
    te["label"] = te["label"].astype("int64")
    ds = Dataset.from_pandas(te, preserve_index=False).cast_column("label", Value("int64"))

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=max_length)

    ds = ds.map(tokenize, batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")
    return ds

def softmax_2d(logits: np.ndarray) -> np.ndarray:
    # 稳定版 softmax
    x = logits - logits.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=1, keepdims=True)

# baseline：用 tweet_clean
ds_base = build_pred_dataset(test_with_topics, "tweet_clean", trainer_base.tokenizer, max_length=128)
pred_base = trainer_base.predict(ds_base)
logits_base = pred_base.predictions
proba_base = softmax_2d(logits_base)
yhat_base = np.argmax(logits_base, axis=1)
p1_base = proba_base[:, 1]  # 预测为 hate(=1) 的概率

# with-topic：用 text_for_bert
ds_topic = build_pred_dataset(test_with_topics, "text_for_bert", trainer_topic.tokenizer, max_length=128)
pred_topic = trainer_topic.predict(ds_topic)
logits_topic = pred_topic.predictions
proba_topic = softmax_2d(logits_topic)
yhat_topic = np.argmax(logits_topic, axis=1)
p1_topic = proba_topic[:, 1]

# 组装对比输出
out = test_with_topics.copy()
out["pred_baseline"] = yhat_base
out["pred_with_topic"] = yhat_topic
out["p_hate_baseline"] = p1_base
out["p_hate_with_topic"] = p1_topic

out["baseline_correct"] = (out["pred_baseline"] == out["label"])
out["with_topic_correct"] = (out["pred_with_topic"] == out["label"])
out["pred_changed"] = (out["pred_baseline"] != out["pred_with_topic"])

# 哪些样本被纠正 / 被带偏
out["corrected_by_topic"] = (~out["baseline_correct"]) & (out["with_topic_correct"])
out["hurt_by_topic"] = (out["baseline_correct"]) & (~out["with_topic_correct"])

# 可选：只保留你关心的列（你也可以注释掉这段保留全部列）
cols_keep = []
for c in ["tweet", "tweet_clean", "text_for_bert", "label", "topic_id", "topic_name", "topic_repr"]:
    if c in out.columns:
        cols_keep.append(c)

cols_keep += [
    "pred_baseline", "pred_with_topic",
    "p_hate_baseline", "p_hate_with_topic",
    "baseline_correct", "with_topic_correct",
    "pred_changed", "corrected_by_topic", "hurt_by_topic"
]
out = out[cols_keep]

out.to_csv("test_predictions_compare_baseline_vs_topic.csv", index=False, encoding="utf-8-sig")
print("Saved: test_predictions_compare_baseline_vs_topic.csv")

# 额外：快速看“纠正/带偏”数量
print("Corrected by topic:", int(out["corrected_by_topic"].sum()))
print("Hurt by topic:", int(out["hurt_by_topic"].sum()))
print("Pred changed:", int(out["pred_changed"].sum()))


                         run  eval_loss  accuracy  f1_macro  runtime_s  \
0      baseline(tweet_clean)   0.609701  0.748954  0.736479     0.8639   
1  with_topic(text_for_bert)   0.617752  0.771967  0.764154     0.8681   

   samples_per_second  
0             553.326  
1             550.605  
Saved: metrics_compare_baseline_vs_topic.csv


Map: 100%|██████████| 478/478 [00:00<00:00, 10959.51 examples/s]


Saved: test_predictions_compare_baseline_vs_topic.csv
Corrected by topic: 24
Hurt by topic: 13
Pred changed: 37


In [20]:
import pandas as pd
df = pd.read_csv("test_predictions_compare_baseline_vs_topic.csv")
df1 = df[df["pred_changed"] == True]
df1.to_csv("pred_changed.csv", index=False)
print(len(df1))


37


### （修改版）仅在train信息后concat

In [5]:
import re
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from umap import UMAP

def clean_tweet(t: str) -> str:
    t = re.sub(r"http\S+|www\.\S+", " ", str(t))
    t = re.sub(r"@\w+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_df = train_df.copy()
test_df  = test_df.copy()

train_df["tweet_clean"] = train_df["tweet"].apply(clean_tweet)
test_df["tweet_clean"]  = test_df["tweet"].apply(clean_tweet)

docs_train = train_df["tweet_clean"].tolist()

umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=2
)
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=1,
    cluster_selection_method="eom",
    prediction_data=True
)
# 向量器（你原来的参数）
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

# BERTopic：只在 train 上拟合
topic_model = BERTopic(
    language="english",
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)

topics_train, _ = topic_model.fit_transform(docs_train)
train_df["topic_id"] = topics_train

# topic -> repr（top words）
topic_info = topic_model.get_topic_info()
TOP_N = 15
topicid_to_repr = {}
for tid in topic_info["Topic"].tolist():
    if tid == -1:
        topicid_to_repr[tid] = ""
        continue
    ww = topic_model.get_topic(tid) or []
    topicid_to_repr[tid] = ", ".join([w for (w, _) in ww[:TOP_N]])

train_df["topic_repr"] = train_df["topic_id"].map(topicid_to_repr)

# 只在 train 上 concat
train_df["text_base"] = train_df["tweet_clean"]
train_df["text_with_topic"] = train_df.apply(
    lambda r: r["tweet_clean"] if (r["topic_id"] == -1 or r["topic_repr"] == "")
    else f'{r["tweet_clean"]} [TOPIC] {r["topic_repr"]}',
    axis=1
)
#cols = ["tweet", "tweet_clean", "label", "topic_id", "topic_name", "topic_repr", "text_for_bert"]
train_df_out = train_df.copy()
train_df_out.to_csv("WS(train)_with_topics.csv", index=False, encoding="utf-8-sig")
print("Saved: WS(train)_with_topics.csv")

print("Train topics:", train_df["topic_id"].nunique(), "(-1 included)")


2026-02-05 19:13:42,180 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 320/320 [00:04<00:00, 79.80it/s] 
2026-02-05 19:13:49,782 - BERTopic - Embedding - Completed ✓
2026-02-05 19:13:49,783 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-05 19:14:19,948 - BERTopic - Dimensionality - Completed ✓
2026-02-05 19:14:19,950 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-05 19:14:20,468 - BERTopic - Cluster - Completed ✓
2026-02-05 19:14:20,475 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-05 19:14:20,691 - BERTopic - Representation - Completed ✓


Saved: WS(train)_with_topics.csv
Train topics: 47 (-1 included)


In [6]:
train_df.head(3)

,tweet,label,tweet_clean,topic_id,topic_repr,text_base,text_with_topic
0,"As of March 13th , 2014 , the booklet had been...",0,"As of March 13th , 2014 , the booklet had been...",19,"book, read, books, pdf, booklet, copy, downloa...","As of March 13th , 2014 , the booklet had been...","As of March 13th , 2014 , the booklet had been..."
1,In order to help increase the booklets downloa...,0,In order to help increase the booklets downloa...,11,"youtube, video, videos, watch, youtube youtube...",In order to help increase the booklets downloa...,In order to help increase the booklets downloa...
2,( Simply copy and paste the following text int...,0,( Simply copy and paste the following text int...,11,"youtube, video, videos, watch, youtube youtube...",( Simply copy and paste the following text int...,( Simply copy and paste the following text int...


In [11]:
import numpy as np
import pandas as pd
import evaluate
from datasets import Dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)


# 0) 固定随机种子：让结果更可复现（可选但强烈推荐）
set_seed(13)


accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
recall_metric = evaluate.load("recall")   # ✅ 新增
precision_metric = evaluate.load("precision")  # （可选，但强烈推荐一起报告）

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    preds = np.argmax(logits, axis=-1)

    # 1) overall
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]

    # 2) recall
    # HATE=1 的 recall（你最关心的：是否检出 hate speech）
    recall_hate = recall_metric.compute(predictions=preds, references=labels, average="binary", pos_label=1)["recall"]

    # 两类 macro recall（每类 recall 平均；不受类不平衡影响）
    recall_macro = recall_metric.compute(predictions=preds, references=labels, average="macro")["recall"]

    # （可选）如果你也想报告 precision（通常和 recall 配套）
    precision_hate = precision_metric.compute(predictions=preds, references=labels, average="binary", pos_label=1)["precision"]

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "recall_hate": recall_hate,       
        "recall_macro": recall_macro,     
        "precision_hate": precision_hate, 
    }



#训练函数：支持 train/test 用不同文本列

def train_bert_classifier(
    train_df_in: pd.DataFrame,
    test_df_in: pd.DataFrame,
    train_text_col: str = "tweet_clean",
    test_text_col: str = "tweet_clean",
    out_dir: str = "bert_out",
    model_name: str = "bert-base-uncased",
    max_length: int = 256,
    epochs: int = 10,
    topic_token = "[TOPIC]"
):
    """
    train_text_col: 训练时用哪一列作为输入文本
    test_text_col : 评估时用哪一列作为输入文本（方式A里通常是 tweet_clean）
    """

    # (a) 取出 text + label，并统一列名为 text
    tr = train_df_in[[train_text_col, "label"]].rename(columns={train_text_col: "text"}).copy()
    te = test_df_in[[test_text_col, "label"]].rename(columns={test_text_col: "text"}).copy()

    # label 强制为 int64
    tr["label"] = tr["label"].astype("int64")
    te["label"] = te["label"].astype("int64")

    # (b) 转 HF Dataset
    train_ds = Dataset.from_pandas(tr, preserve_index=False).cast_column("label", Value("int64"))
    test_ds  = Dataset.from_pandas(te, preserve_index=False).cast_column("label", Value("int64"))

    # (c) tokenizer + 动态 padding（比 padding=max_length 更省显存）
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
    test_ds  = test_ds.map(tokenize, batched=True, remove_columns=["text"])

    train_ds.set_format("torch")
    test_ds.set_format("torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # (d) model + resize embedding
    TOPIC_TOKEN = "[TOPIC]"
    num_added = tokenizer.add_special_tokens({"additional_special_tokens":[TOPIC_TOKEN]})
    print("Added special tokens:",num_added)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    if num_added > 0:
        model.resize_token_embeddings(len(tokenizer))
    
    
    # (e) training args
    # Trainer 会在 torch.cuda.is_available()==True 时自动用 GPU
    args = TrainingArguments(
        output_dir=out_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        report_to="none",
        fp16=True,  # 如果你 GPU 可用，fp16 通常更省显存更快；不想用可改 False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    save_dir = os.path.join("models_topic",out_dir)
    os.makedirs(save_dir,exist_ok=True)
    
    # (f) train + eval
    trainer.train()
    metrics = trainer.evaluate()
    
    tokenizer.save_pretrained(save_dir)
    trainer.save_model(save_dir)

    


    # (g) 预测（用于逐条对比）
    pred_out = trainer.predict(test_ds)
    logits = pred_out.predictions
    if isinstance(logits, (tuple, list)):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)

    return trainer, metrics, preds

# -----------------------
# 3) 运行对比：Baseline vs Topic-aug(train only)
# -----------------------
# 你需要确保：
# - train_df 有列：tweet_clean, label, text_with_topic
# - test_df  有列：tweet_clean, label

# Baseline：train=tweet_clean, test=tweet_clean
trainer_base, metrics_base, preds_base = train_bert_classifier(
    train_df, test_df,
    train_text_col="tweet_clean",
    test_text_col="tweet_clean",
    out_dir="bert_baseline"
)

# Topic-aug：train=text_with_topic, test仍=tweet_clean（方式A）
trainer_topic, metrics_topic, preds_topic = train_bert_classifier(
    train_df, test_df,
    train_text_col="text_with_topic",
    test_text_col="tweet_clean",
    out_dir="bert_with_topics_trainonly"
)

# -----------------------
# 4) 指标对比表
# -----------------------
compare_metrics = pd.DataFrame([
    {"run": "baseline", **{k: metrics_base[k] for k in metrics_base if k.startswith("eval_")}},
    {"run": "train_with_topic_test_clean", **{k: metrics_topic[k] for k in metrics_topic if k.startswith("eval_")}},
])
print(compare_metrics)

# -----------------------
# 5) 逐条写回 CSV：看哪些样本被纠正/被带偏
# -----------------------
out = test_df.copy()
out["pred_baseline"] = preds_base
out["pred_topic_trainonly"] = preds_topic
out["correct_baseline"] = (out["pred_baseline"] == out["label"]).astype(int)
out["correct_topic_trainonly"] = (out["pred_topic_trainonly"] == out["label"]).astype(int)

# 标记变化类型
def tag_change(r):
    b = r["correct_baseline"]
    t = r["correct_topic_trainonly"]
    if b == 0 and t == 1:
        return "fixed_by_topic"
    if b == 1 and t == 0:
        return "broken_by_topic"
    if b == 1 and t == 1:
        return "both_correct"
    return "both_wrong"

out["change_tag"] = out.apply(tag_change, axis=1)

# 保存
out.to_csv("compare_baseline_vs_topictrainonly.csv", index=False, encoding="utf-8-sig")
compare_metrics.to_csv("metrics_compare.csv", index=False, encoding="utf-8-sig")

print("Saved: compare_baseline_vs_topictrainonly.csv, metrics_compare.csv")


Map: 100%|██████████| 478/478 [00:00<00:00, 28546.90 examples/s]


Added special tokens: 1


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Recall Hate,Recall Macro,Precision Hate
1,0.243500,0.706957,0.694561,0.669345,0.418410,0.694561,0.934579


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("models_topic/bert_with_topics_trainonly")
model = AutoModelForSequenceClassification.from_pretrained("models_topic/bert_with_topics_trainonly")

# 只对 test 中 label=1 的文本分配 hate-topic
test_df = test_df.copy()
test_df["hate_topic_id"] = -100

mask_hate = test_df["label"] == 1
docs_test_hate = test_df.loc[mask_hate, "tweet_clean"].astype(str).tolist()

topics_test, _ = hate_topic_model.transform(docs_test_hate)
test_df.loc[mask_hate, "hate_topic_id"] = topics_test


NameError: name 'hate_topic_model' is not defined

### 计算topic熵 以及topic对判断的重要度

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("models_topic/bert_with_topics_trainonly")
model = AutoModelForSequenceClassification.from_pretrained("models_topic/bert_with_topics_trainonly")


In [14]:
###计算topic token的熵（Entropy by EAR）

import torch
import numpy as np

TOPIC_TOKEN = "[TOPIC]"          # 你用的 marker
MAX_LEN = 256
BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def build_topic_mask(input_ids, attention_mask, topic_token_id):
    """
    topic_mask[b, j]=True 表示第 b 条样本的第 j 个 token 属于 [TOPIC] 后面的 topic 段（不含 marker）
    """
    B, S = input_ids.shape
    topic_mask = torch.zeros((B, S), dtype=torch.bool, device=input_ids.device)
    for b in range(B):
        pos = (input_ids[b] == topic_token_id).nonzero(as_tuple=True)[0]
        if len(pos) == 0:
            continue
        start = int(pos[0].item()) + 1
        # topic 段 = marker 后面所有非 padding token
        topic_mask[b, start:] = attention_mask[b, start:].bool()
    return topic_mask

@torch.no_grad()
def topic_token_attention_entropy(model, tokenizer, texts):
    model.eval()
    model.to(DEVICE)

    topic_token_id = tokenizer.convert_tokens_to_ids(TOPIC_TOKEN)
    if topic_token_id == tokenizer.unk_token_id:
        raise ValueError(f"{TOPIC_TOKEN} 似乎没被 tokenizer 识别为单独token。请确认已 add_special_tokens 并保存/加载一致。")

    results = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = [str(t) for t in texts[i:i+BATCH_SIZE]]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        ).to(DEVICE)

        out = model(**enc, output_attentions=True)
        # 取最后一层 attentions: [B, H, S, S]
        att = out.attentions[-1]

        # HF 的 attention 一般已经是 softmax 后的概率；head mean 后仍是概率分布（每行和≈1）
        p = att.mean(dim=1)  # [B, S, S]
        p = p / p.sum(dim=-1, keepdim=True).clamp(min=1e-12)  # 保险归一化

        # token-level entropy: H_i = - sum_j p_ij log p_ij  -> [B, S]
        H = -(p * torch.log(p.clamp(min=1e-12))).sum(dim=-1)

        # 去掉 padding（论文要求 discard padding）
        valid = enc["attention_mask"].bool()  # [B, S]
        topic_mask = build_topic_mask(enc["input_ids"], enc["attention_mask"], topic_token_id)
        topic_mask = topic_mask & valid

        # 对 topic tokens 的熵取平均：每条样本一个数
        denom = topic_mask.sum(dim=1).clamp(min=1)              # [B]
        topic_entropy = (H * topic_mask).sum(dim=1) / denom     # [B]

        results.extend(topic_entropy.detach().cpu().numpy().tolist())

        # 防止显存堆积
        del out, att, p, H

    return np.array(results)

#import numpy as np
#import pandas as pd

#texts = train_df["text_with_topic"].astype(str).tolist()
#ent = topic_token_attention_entropy(model, tokenizer, texts)  # [N]

#df = train_df.copy()
#df["topic_entropy"] = ent

# 过滤：没 [TOPIC] 的样本在你的实现里 entropy=0（你之前也遇到过）
#df_valid = df[df["topic_entropy"] > 0].copy()

#topic_entropy_table = (
#    df_valid.groupby("topic_id")["topic_entropy"]
#    .agg(n="count", mean="mean", median="median", std="std")
#    .reset_index()
#    .sort_values("mean", ascending=False)
#)

#print(topic_entropy_table.head(20))


BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


     topic_id   n      mean    median       std
152       152  12  2.606228  2.604739  0.172138
8           8  87  2.593919  2.564844  0.180575
98         98  20  2.581593  2.545511  0.287426
82         82  22  2.545848  2.552362  0.237273
43         43  36  2.534479  2.494155  0.161602
162       162  11  2.532749  2.497599  0.122556
104       104  19  2.532532  2.514973  0.132788
25         25  47  2.509869  2.487486  0.235571
164       164  11  2.501518  2.525033  0.226566
24         24  49  2.487978  2.411739  0.200301
73         73  25  2.484317  2.414628  0.158660
64         64  28  2.477387  2.456323  0.171907
131       131  15  2.472027  2.427497  0.154358
149       149  13  2.467829  2.401586  0.265906
150       150  13  2.462716  2.435856  0.093200
129       129  16  2.452364  2.484443  0.170629
119       119  17  2.449208  2.422609  0.161248
56         56  31  2.444022  2.464855  0.253309
122       122  16  2.439115  2.465781  0.111212
114       114  17  2.439095  2.462298  0

In [16]:
import numpy as np
import pandas as pd

# 1) 逐样本计算 topic-token entropy
texts = train_df["text_with_topic"].astype(str).tolist()
ent = topic_token_attention_entropy(model, tokenizer, texts)  # 你已有的函数

df = train_df.copy()
df["topic_entropy"] = ent

# 2) 过滤：没有 [TOPIC] 的样本会得到 0（你之前也提到过）
df_valid = df[df["topic_entropy"] > 0].copy()

# 3) 生成 topic_repr_top10（把 topic_repr 截取前10个关键词组）
def top10_phrases(topic_repr):
    if pd.isna(topic_repr) or str(topic_repr).strip() == "":
        return "OUTLIER"
    parts = [p.strip() for p in str(topic_repr).split(",") if p.strip()]
    return ", ".join(parts[:10])

df_valid["topic_repr_top10"] = df_valid["topic_repr"].apply(top10_phrases)

# 4) 方案A：按 topic_id 聚合 entropy
agg_stats = (
    df_valid.groupby("topic_id")["topic_entropy"]
    .agg(n="count", mean="mean", median="median", std="std")
    .reset_index()
)

# 5) 把每个 topic_id 对应的 topic_repr_top10 拼回去（取该topic出现的第一个非空repr）
agg_repr = (
    df_valid.groupby("topic_id")["topic_repr_top10"]
    .agg(lambda s: next((x for x in s if isinstance(x, str) and x.strip() and x != "OUTLIER"), "OUTLIER"))
    .reset_index()
)

topic_entropy_table = agg_stats.merge(agg_repr, on="topic_id", how="left")

# 6) 排序（例如按 mean 从高到低）
topic_entropy_table = topic_entropy_table.sort_values("mean", ascending=False).reset_index(drop=True)
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", None,
):
    print(topic_entropy_table.head(20))




    topic_id   n      mean    median       std  \
0        152  12  2.606228  2.604739  0.172138   
1          8  87  2.593919  2.564844  0.180575   
2         98  20  2.581593  2.545511  0.287426   
3         82  22  2.545848  2.552362  0.237273   
4         43  36  2.534479  2.494155  0.161602   
5        162  11  2.532749  2.497599  0.122556   
6        104  19  2.532532  2.514973  0.132788   
7         25  47  2.509869  2.487486  0.235571   
8        164  11  2.501518  2.525033  0.226566   
9         24  49  2.487978  2.411739  0.200301   
10        73  25  2.484317  2.414628  0.158660   
11        64  28  2.477387  2.456323  0.171907   
12       131  15  2.472027  2.427497  0.154358   
13       149  13  2.467829  2.401586  0.265906   
14       150  13  2.462716  2.435856  0.093200   
15       129  16  2.452364  2.484443  0.170629   
16       119  17  2.449208  2.422609  0.161248   
17        56  31  2.444022  2.464855  0.253309   
18       122  16  2.439115  2.465781  0.111212   


In [17]:
topic_entropy_table.head(20)

,topic_id,n,mean,median,std,topic_repr_top10
0,152,12,2.606228,2.604739,0.172138,"shoulder, actually good, 828, welcome come, sh..."
1,8,87,2.593919,2.564844,0.180575,"whites, minority, white people, white, white r..."
2,98,20,2.581593,2.545511,0.287426,"image, aug, sacred, center, images, light, pla..."
3,82,22,2.545848,2.552362,0.237273,"genetic, half white, half, tend, white blood, ..."
4,43,36,2.534479,2.494155,0.161602,"youtube black, youtube, black, youtube white, ..."
5,162,11,2.532749,2.497599,0.122556,"great idea, need stop, stop talking, want way,..."
6,104,19,2.532532,2.514973,0.132788,"race, matters, love race, help, love, destroy ..."
7,25,47,2.509869,2.487486,0.235571,"looking, im looking, im, female, email, hi, me..."
8,164,11,2.501518,2.525033,0.226566,"germans, germany, hitler, champion, message pe..."
9,24,49,2.487978,2.411739,0.200301,"white kids, kids, white, children, white babie..."


In [33]:
import torch
import numpy as np

TOPIC_TOKEN = "[TOPIC]"
MAX_LEN = 256
BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def build_topic_mask(input_ids, attention_mask, topic_token_id):
    B, S = input_ids.shape
    topic_mask = torch.zeros((B, S), dtype=torch.bool, device=input_ids.device)
    for b in range(B):
        pos = (input_ids[b] == topic_token_id).nonzero(as_tuple=True)[0]
        if len(pos) == 0:
            continue
        start = int(pos[0].item()) + 1
        topic_mask[b, start:] = attention_mask[b, start:].bool()
    return topic_mask

@torch.no_grad()
def cls_to_topic_attention_mass(model, tokenizer, texts):
    model.eval()
    model.to(DEVICE)

    topic_token_id = tokenizer.convert_tokens_to_ids(TOPIC_TOKEN)
    if topic_token_id == tokenizer.unk_token_id:
        raise ValueError(f"{TOPIC_TOKEN} 似乎没被 tokenizer 识别为单独token。请确认已 add_special_tokens 并保存/加载一致。")

    results = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = [str(t) for t in texts[i:i+BATCH_SIZE]]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        ).to(DEVICE)

        out = model(**enc, output_attentions=True)
        att = out.attentions[-1]         # [B, H, S, S]
        att_mean = att.mean(dim=1)       # [B, S, S]，每行和≈1

        topic_mask = build_topic_mask(enc["input_ids"], enc["attention_mask"], topic_token_id)  # [B,S]

        # CLS 在序列第 0 位：取 CLS 的那一行，对 topic 列求和 -> [B]
        cls_row = att_mean[:, 0, :]                          # [B, S]
        cls_to_topic = (cls_row * topic_mask.float()).sum(dim=1)  # [B]

        results.extend(cls_to_topic.detach().cpu().numpy().tolist())

        del out, att, att_mean

    return np.array(results)

# ===== 用法示例 =====
texts = train_df["text_with_topic"].astype(str).tolist()
mass = cls_to_topic_attention_mass(model, tokenizer, texts)
print(mass[:10], mass.mean())


[0.33672833 0.         0.         0.         0.44126791 0.4031536
 0.41716552 0.         0.         0.        ] 0.1497387228776817


### 关联分析

In [19]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def predict_proba_and_pred(model, tokenizer, texts, max_len=256, batch_size=16):
    model.eval().to(DEVICE)
    probs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i+batch_size]]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        out = model(**enc)
        logits = out.logits  # [B, 2] 假设二分类
        p1 = torch.softmax(logits, dim=-1)[:, 1]  # P(y=1)
        probs.extend(p1.detach().cpu().numpy().tolist())
    probs = np.array(probs)
    pred = (probs >= 0.5).astype(int)
    return probs, pred

# === 得到每条样本预测 ===
texts = train_df["text_with_topic"].astype(str).tolist()
proba1, pred = predict_proba_and_pred(model, tokenizer, texts)

df = train_df.copy()
df["proba1"] = proba1
df["pred"] = pred


In [20]:
ent = topic_token_attention_entropy(model, tokenizer, texts)  # [N]
df["topic_entropy"] = ent

# 过滤：没有[TOPIC]导致entropy=0的样本（你之前不想纳入平均）
df_entropy_valid = df[df["topic_entropy"] > 0].copy()


In [21]:
def safe_prf(y_true, y_pred):
    # 返回正类(1)的 precision/recall/f1；若全为单一类，设为 NaN 更合理
    if len(np.unique(y_true)) < 2:
        return (np.nan, np.nan, np.nan)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    return (p, r, f1)

rows = []
for tid, g in df.groupby("topic_id"):
    y_true = g["label"].astype(int).values
    y_pred = g["pred"].astype(int).values

    n = len(g)
    pos_rate = y_true.mean()

    # confusion matrix (TN FP / FN TP)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    prec, rec, f1 = safe_prf(y_true, y_pred)
    acc = (y_true == y_pred).mean()

    # topic_repr: 取该topic第一条非空
    topic_repr = next((x for x in g["topic_repr"].astype(str).tolist() if x.strip()), "")

    # entropy：用过滤后的 df_entropy_valid 来算 topic 的 mean（避免 0 污染）
    ge = df_entropy_valid[df_entropy_valid["topic_id"] == tid]["topic_entropy"].values
    ent_mean = float(np.mean(ge)) if len(ge) else np.nan

    rows.append({
        "topic_id": tid,
        "n": n,
        "pos_rate": pos_rate,
        "acc": acc,
        "precision_1": prec,
        "recall_1": rec,
        "f1_1": f1,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "entropy_mean": ent_mean,
        "topic_repr": topic_repr
    })

topic_perf = pd.DataFrame(rows)

# topic_repr 取前10个关键词
def top10_phrases(topic_repr):
    if pd.isna(topic_repr) or str(topic_repr).strip() == "":
        return "OUTLIER"
    parts = [p.strip() for p in str(topic_repr).split(",") if p.strip()]
    return ", ".join(parts[:10])

topic_perf["topic_repr_top10"] = topic_perf["topic_repr"].apply(top10_phrases)

# 排序：比如看 pos_rate 高的 topic，模型 F1 如何
topic_perf = topic_perf.sort_values(["pos_rate", "n"], ascending=[False, False]).reset_index(drop=True)

topic_perf[["topic_id","n","pos_rate","f1_1","precision_1","recall_1","fp","fn","entropy_mean","topic_repr_top10"]].head(20)


,topic_id,n,pos_rate,f1_1,precision_1,recall_1,fp,fn,entropy_mean,topic_repr_top10
0,29,45,0.777778,0.875000,0.777778,1.000000,10,0,2.261060,"ape, apes, monkeys, monkey, planet, subhuman, ..."
1,134,15,0.666667,0.800000,0.666667,1.000000,5,0,2.412396,"groid, white woman, imagine, woman, came runni..."
2,112,18,0.500000,1.000000,1.000000,1.000000,0,0,2.322387,"welfare, money, food stamps, housing, stamps, ..."
3,150,13,0.461538,1.000000,1.000000,1.000000,0,0,2.462716,"pakistani, pakis, celebrating, pakistan, laugh..."
4,5,127,0.440945,0.818898,0.732394,0.928571,19,4,2.130180,"jews, jew, jewish, jesus, bible, word, chosen,..."
5,0,200,0.430000,0.836957,0.785714,0.895349,21,9,2.350589,"blacks, black, negro, negroes, black people, s..."
6,54,31,0.354839,0.782609,0.750000,0.818182,3,2,2.098071,"gay, homosexuals, gays, homosexual, homosexual..."
7,22,51,0.333333,0.894737,0.809524,1.000000,4,0,2.085286,"land, killing, invaded, feds, weak, labor, afr..."
8,18,57,0.315789,0.857143,0.882353,0.833333,2,3,2.086361,"africa, south africa, south, sa, nigeria, afri..."
9,63,29,0.310345,0.842105,0.800000,0.888889,2,1,1.999286,"israel, zionist, israeli, intentional, western..."


In [22]:
import numpy as np
import pandas as pd

def trainer_predict_binary(trainer, dataset):
    """
    返回:
      proba1: P(y=1) (N,)
      pred:  0/1 (N,)
      y_true: (N,)
    """
    out = trainer.predict(dataset)
    logits = out.predictions              # (N, 2) 二分类
    y_true = out.label_ids.astype(int)    # (N,)

    # softmax -> P(y=1)
    exp = np.exp(logits - logits.max(axis=1, keepdims=True))
    proba = exp / exp.sum(axis=1, keepdims=True)
    proba1 = proba[:, 1]

    pred = (proba1 >= 0.5).astype(int)
    return proba1, pred, y_true


In [23]:
proba1, pred, y_true = trainer_predict_binary(trainer, test_dataset)

NameError: name 'trainer' is not defined

### 各hate Topic内表现

In [10]:
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# 你已有：all_df（包含 tweet_clean, label），以及 vectorizer_model

umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=2
)
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=1,
    cluster_selection_method="eom",
    prediction_data=True
)

hate_df = all_df[all_df["label"] == 1].copy()   # 这里别 reset_index(drop=True)，方便写回
docs_hate = hate_df["tweet_clean"].astype(str).tolist()

topic_model_hate = BERTopic(
    language="english",
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=False,
    verbose=True
)

topics_hate, _ = topic_model_hate.fit_transform(docs_hate)
hate_df["topic_hate_only"] = topics_hate

# 写回 all_df：label=0 的 topic 用 -100 占位
all_df = all_df.copy()
all_df["topic_hate_only"] = -100
all_df.loc[hate_df.index, "topic_hate_only"] = hate_df["topic_hate_only"]

# 如果你有独立 test_df，并且它的行在 all_df 里可对应：
# 最稳：保证 all_df 是 train_df + test_df concat 得到的，然后这样切回去：
test_df2 = all_df.iloc[len(train_df):].copy()   # train_df 在前，test_df 在后


2026-01-31 17:31:12,364 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 38/38 [00:00<00:00, 114.99it/s]
2026-01-31 17:31:15,065 - BERTopic - Embedding - Completed ✓
2026-01-31 17:31:15,065 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-31 17:31:23,006 - BERTopic - Dimensionality - Completed ✓
2026-01-31 17:31:23,008 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-31 17:31:23,051 - BERTopic - Cluster - Completed ✓
2026-01-31 17:31:23,053 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-31 17:31:23,093 - BERTopic - Representation - Completed ✓


In [11]:
TOP_N = 15
topic_info = topic_model_hate.get_topic_info()
tid2repr = {}
for tid in topic_info["Topic"].tolist():
    if tid == -1:
        tid2repr[tid] = ""
    else:
        ww = topic_model_hate.get_topic(tid) or []
        tid2repr[tid] = ", ".join([w for (w, _) in ww[:TOP_N]])

test_df2["topic_repr"] = test_df2["topic_hate_only"].map(tid2repr).fillna("")


In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("models_topic/bert_with_topics_trainonly")
model = AutoModelForSequenceClassification.from_pretrained("models_topic/bert_with_topics_trainonly")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30523, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [17]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

@torch.no_grad()
def predict_texts(texts, batch_size=64, max_length=256):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        preds.append(torch.argmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(preds)

test_df2["pred"] = predict_texts(test_df2["tweet_clean"].astype(str).tolist())


In [20]:
import pandas as pd

sub = test_df2[(test_df2["label"] == 1) & (test_df2["topic_hate_only"] >= 0)].copy()

topic_recall = (
    sub.groupby("topic_hate_only")
       .agg(
            n_samples=("label", "size"),
            recall_hate=("pred", "mean"),   # 因为 y_true 全是 1，所以 recall = mean(pred==1)
            miss_rate=("pred", lambda x: 1 - x.mean()),
            topic_repr=("topic_repr", "first")
        )
       .reset_index()
       .rename(columns={"topic_hate_only": "topic_id"})
)

topic_recall = topic_recall.query("n_samples >= 5").sort_values("recall_hate")  # 阈值可调
print(topic_recall.head(30))


   topic_id  n_samples  recall_hate  miss_rate  \
5         5         14     0.357143   0.642857   
4         4         19     0.421053   0.578947   
1         1         24     0.500000   0.500000   
7         7         12     0.500000   0.500000   
6         6         15     0.533333   0.466667   
0         0         34     0.617647   0.382353   
2         2         27     0.666667   0.333333   
3         3         23     0.782609   0.217391   

                                          topic_repr  
5  school, white, asian, race, kids, fight, mixin...  
4  just, want, nt, welfare, hopefully, job, till,...  
1  jews, jew, jewish, people, whites, zionist, ju...  
7  london, canada, white, whites, city, left, liv...  
6  sweden, muslim, scum, ing, europe, islam, just...  
0  black, negro, negroes, blacks, white, nt, like...  
2  whites, africa, white, blacks, country, nonwhi...  
3  apes, groid, monkeys, monkey, going, just, dau...  


In [19]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

rows = []
for tid, g in test_df2.groupby("topic_hate_only"):
    y_true = g["label"].values
    y_pred = g["pred"].values

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    rows.append({
        "topic_id": tid,
        "n_samples": len(g),
        "accuracy": acc,
        "precision_hate": p,
        "recall_hate": r,
        "f1_hate": f1,
        "pos_rate_true": float(y_true.mean()),
        "pos_rate_pred": float(y_pred.mean()),
        "topic_repr": (g["topic_repr"].iloc[0] if "topic_repr" in g.columns else "")
    })

topic_metrics = pd.DataFrame(rows).sort_values(["f1_hate", "n_samples"], ascending=[True, False])
print(topic_metrics.head(30))


   topic_id  n_samples  accuracy  precision_hate  recall_hate   f1_hate  \
0      -100        239  0.953975             0.0     0.000000  0.000000   
7         5         14  0.357143             1.0     0.357143  0.526316   
6         4         19  0.421053             1.0     0.421053  0.592593   
1        -1         71  0.422535             1.0     0.422535  0.594059   
3         1         24  0.500000             1.0     0.500000  0.666667   
9         7         12  0.500000             1.0     0.500000  0.666667   
8         6         15  0.533333             1.0     0.533333  0.695652   
2         0         34  0.617647             1.0     0.617647  0.763636   
4         2         27  0.666667             1.0     0.666667  0.800000   
5         3         23  0.782609             1.0     0.782609  0.878049   

   pos_rate_true  pos_rate_pred  \
0            0.0       0.046025   
7            1.0       0.357143   
6            1.0       0.421053   
1            1.0       0.422535   